In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/offside-data-thon/sample_submission.csv
/kaggle/input/competitions/offside-data-thon/data_dictionary.csv
/kaggle/input/competitions/offside-data-thon/feature_catalog.csv
/kaggle/input/competitions/offside-data-thon/train.csv
/kaggle/input/competitions/offside-data-thon/test.csv


In [3]:
train = pd.read_csv('/kaggle/input/competitions/offside-data-thon/train.csv')
test = pd.read_csv('/kaggle/input/competitions/offside-data-thon/test.csv')
train.head()

/tmp/ipykernel_58/326530912.py:1: DtypeWarning: Columns (39,58) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/kaggle/input/competitions/offside-data-thon/train.csv')


,appearance_id,date,season,minutes_played,yellow_cards,red_cards,disciplinary_points,home_club_id,away_club_id,home_club_name,...,has_national_team_experience,xG_to_xA_ratio,finisher_flag,creative_player_flag,analytics_coverage_flag,is_goalkeeper,is_defender,is_midfielder,is_attacker,name_y
0,2335635_20137,2013-07-15,2013,90,0,0,0,6414,2783,Metalist Kharkiv (- 2016),...,True,0.362550,False,True,True,False,False,True,False,José Sosa
1,2654328_36235,2016-03-07,2015,90,0,0,0,2700,4128,Anzhi Makhachkala ( -2022),...,False,0.454626,False,True,True,False,True,False,False,Petar Zanev
2,4098806_1163778,2024-03-03,2023,63,0,0,0,1096,475,Royal Antwerp Football Club,...,False,NaN,False,False,False,False,False,False,True,Kahveh Zahiroleslam
3,2700658_348863,2016-08-28,2016,33,2,0,2,383,202,Eindhovense Voetbalvereniging Philips Sport Ve...,...,True,0.569382,False,True,True,False,False,True,False,Juninho Bacuna
4,3016404_240551,2018-03-31,2017,65,0,0,0,4128,932,Amkar Perm,...,False,NaN,False,False,False,False,False,False,True,Aaron Olanare


In [4]:
print("train Shape:", train.shape)
print("test Shape:", test.shape)

train Shape: (1319813, 64)
test Shape: (565635, 63)


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1319813 entries, 0 to 1319812
Data columns (total 64 columns):
 #   Column                        Non-Null Count    Dtype  
---  ------                        --------------    -----  
 0   appearance_id                 1319813 non-null  object 
 1   date                          1319813 non-null  object 
 2   season                        1319813 non-null  int64  
 3   minutes_played                1319813 non-null  int64  
 4   yellow_cards                  1319813 non-null  int64  
 5   red_cards                     1319813 non-null  int64  
 6   disciplinary_points           1319813 non-null  int64  
 7   home_club_id                  1319813 non-null  int64  
 8   away_club_id                  1319813 non-null  int64  
 9   home_club_name                1319719 non-null  object 
 10  away_club_name                1319812 non-null  object 
 11  home_club_goals               1319813 non-null  int64  
 12  away_club_goals             

In [6]:
train['scored_flag'].value_counts(normalize=True)

scored_flag
False    0.914623
True     0.085377
Name: proportion, dtype: float64

In [7]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['scored_flag'])
y = train['scored_flag']

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)

(1055850, 63)
(263963, 63)


In [8]:
from catboost import CatBoostClassifier

In [9]:
cat_cols = X_train.select_dtypes(
    include=['object']
).columns.tolist()

print(cat_cols)

['appearance_id', 'date', 'home_club_name', 'away_club_name', 'home_away', 'stadium', 'referee', 'name_x', 'country_name', 'competition_type', 'confederation', 'foot', 'position', 'sub_position', 'country_of_citizenship', 'has_understat', 'market_value_tier', 'age_bucket', 'analytics_coverage_flag', 'name_y']


In [10]:
train[['name_x','scored_flag']].groupby('name_x').agg(
    matches=('scored_flag','count'),
    scoring_rate=('scored_flag','mean')
).sort_values('matches', ascending=False).head(20)

,matches,scoring_rate
name_x,,
premier-liga,119789,0.077829
serie-a,108482,0.082373
laliga,107296,0.078885
premier-league,104943,0.086104
ligue-1,100303,0.079878
super-lig,92466,0.083977
bundesliga,87166,0.091171
eredivisie,84165,0.094018
liga-portugal,83676,0.078254


In [11]:
train[['name_x','name_y']].head(20)

,name_x,name_y
0,premier-liga,José Sosa
1,premier-liga,Petar Zanev
2,jupiler-pro-league,Kahveh Zahiroleslam
3,eredivisie,Juninho Bacuna
4,premier-liga,Aaron Olanare
5,super-lig,Murat Kayali
6,liga-portugal,Pedro Moreira
7,laliga,Samuel Chukwueze
8,serie-a,Marco Motta
9,dfb-pokal,Christopher Nkunku


In [12]:
player_stats = (
    train.groupby('name_y')['scored_flag']
    .agg(['count', 'mean'])
    .sort_values('count', ascending=False)
)

player_stats.head(20)

,count,mean
name_y,,
Danilo,812,0.089901
Paulinho,778,0.138817
Marcelo,716,0.055866
Guilherme,571,0.038529
João Pedro,568,0.184859
Fernando,507,0.080868
Pedro,479,0.169102
Antoine Griezmann,468,0.320513
Robert Lewandowski,453,0.520971


In [13]:
player_stats.sort_values('mean', ascending=False).head(20)

,count,mean
name_y,,
Iván Azón,1,1.0
Ivan Zazvonkin,1,1.0
Julian Del Rio,1,1.0
Julian Riedel,1,1.0
Julio Villalba,1,1.0
Scott Bright,1,1.0
Danil Klenkin,1,1.0
Shamil Gadzhiev,1,1.0
Julius Voldby,1,1.0


In [14]:
print("Train players:", train['name_y'].nunique())
print("Test players:", test['name_y'].nunique())

overlap = len(set(train['name_y']) & set(test['name_y']))
print("Overlap:", overlap)

Train players: 27282
Test players: 24850
Overlap: 24012


In [15]:
position_stats = (
    train.groupby('position')['scored_flag']
    .agg(['count', 'mean'])
    .sort_values('mean', ascending=False)
)

position_stats

,count,mean
position,,
Attack,372175,0.172706
Missing,1058,0.145558
Midfield,409269,0.077463
Defender,444671,0.037180
Goalkeeper,92638,0.000162


In [16]:
for col in train.columns:
    if "goal" in col.lower():
        print(col)

home_club_goals
away_club_goals
international_goals
goal_diff_abs
goal_per_cap
is_goalkeeper


In [18]:
train['attacking_index'] = (
    train['avg_xG'].fillna(0)
    + train['avg_npxG'].fillna(0)
    + train['avg_shots'].fillna(0)
)

test['attacking_index'] = (
    test['avg_xG'].fillna(0)
    + test['avg_npxG'].fillna(0)
    + test['avg_shots'].fillna(0)
)

train['creative_index'] = (
    train['avg_xA'].fillna(0)
    + train['avg_key_passes'].fillna(0)
)

test['creative_index'] = (
    test['avg_xA'].fillna(0)
    + test['avg_key_passes'].fillna(0)
)

In [20]:
y = train['scored_flag'].astype(int)
X = train.drop(columns=['scored_flag'])

In [21]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    test[col] = test[col].fillna('Missing').astype(str)

In [22]:
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

for col in num_cols:
    median = X[col].median()

    X[col] = X[col].fillna(median)
    test[col] = test[col].fillna(median)

In [23]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    combined = pd.concat([
        X[col],
        test[col]
    ]).astype(str)

    le.fit(combined)

    X[col] = le.transform(X[col])
    test[col] = le.transform(test[col])

    encoders[col] = le

In [25]:
train['name_y'].isna().sum()

np.int64(2)

In [27]:
groups = train['name_y'].fillna('Unknown_Player')

In [28]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

train_idx, valid_idx = next(
    gkf.split(X, y, groups)
)

X_train = X.iloc[train_idx].copy()
X_valid = X.iloc[valid_idx].copy()

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

print(X_train.shape)
print(X_valid.shape)

(1055850, 65)
(263963, 65)


In [29]:
print("Missing player names:", train['name_y'].isna().sum())
print("Unique players:", train['name_y'].nunique())

Missing player names: 2
Unique players: 27282


In [30]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    objective='binary',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Number of positive: 90143, number of negative: 965707
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.170953 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7488
[LightGBM] [Info] Number of data points in the train set: 1055850, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.085375 -> initscore=-2.371463
[LightGBM] [Info] Start training from score -2.371463


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
               n_jobs=-1, num_leaves=63, objective='binary', random_state=42,
               subsample=0.8)

In [31]:
from sklearn.metrics import average_precision_score

preds = model.predict_proba(X_valid)[:, 1]

ap = average_precision_score(
    y_valid,
    preds
)

print("AP Score =", ap)

AP Score = 0.3995550743673381


In [32]:
import pandas as pd

fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
})

fi = fi.sort_values(
    'importance',
    ascending=False
)

fi.head(20)

,feature,importance
3,minutes_played,1328
16,referee,1321
14,attendance,1320
1,date,1175
41,value_pct_of_peak,1144
11,home_club_goals,1139
22,age,1123
8,away_club_id,1089
12,away_club_goals,1081
10,away_club_name,1072


In [34]:
player_rate = (
    pd.DataFrame({
        'player': X_train['name_y'],
        'target': y_train
    })
    .groupby('player')['target']
    .mean()
)

In [35]:
global_rate = y_train.mean()

print(global_rate)

0.08537481649855566


In [39]:
X_train['player_goal_rate'] = (
    X_train['name_y']
    .map(player_rate)
    .fillna(global_rate)
)

In [40]:
X_valid['player_goal_rate'] = (
    X_valid['name_y']
    .map(player_rate)
    .fillna(global_rate)
)

In [41]:
X_train[['name_y', 'player_goal_rate']].head()

,name_y,player_goal_rate
0,13295,0.132420
1,21376,0.008621
2,13844,0.148148
3,13648,0.033708
4,36,0.181818


In [42]:
from lightgbm import LGBMClassifier

model2 = LGBMClassifier(
    objective='binary',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model2.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Number of positive: 90143, number of negative: 965707
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.169611 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7743
[LightGBM] [Info] Number of data points in the train set: 1055850, number of used features: 66
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.085375 -> initscore=-2.371463
[LightGBM] [Info] Start training from score -2.371463


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
               n_jobs=-1, num_leaves=63, objective='binary', random_state=42,
               subsample=0.8)

In [43]:
from sklearn.metrics import average_precision_score

preds2 = model2.predict_proba(X_valid)[:,1]

ap2 = average_precision_score(
    y_valid,
    preds2
)

print("Old AP =", 0.3995550743673381)
print("New AP =", ap2)
print("Gain =", ap2 - 0.3995550743673381)

Old AP = 0.3995550743673381
New AP = 0.3318568512232416
Gain = -0.06769822314409651


In [44]:
print(train[['avg_xG','avg_npxG','avg_shots']].isna().mean())

avg_xG       0.478253
avg_npxG     0.478253
avg_shots    0.478253
dtype: float64


In [45]:
train[['avg_xG','avg_npxG','avg_shots']].describe()

,avg_xG,avg_npxG,avg_shots
count,688608.000000,688608.000000,688608.000000
mean,2.020212,1.836850,18.014880
std,2.727253,2.385092,17.758233
min,0.000000,0.000000,0.000000
25%,0.393977,0.381321,5.666667
50%,1.076438,1.020929,13.000000
75%,2.501710,2.290445,24.666667
max,26.462077,23.371345,162.444444


In [46]:
train.groupby('has_understat')['scored_flag'].mean()

has_understat
False    0.076982
True     0.093073
Name: scored_flag, dtype: float64

In [47]:
train.groupby('analytics_coverage_flag')['scored_flag'].mean()

analytics_coverage_flag
False    0.076982
True     0.093073
Name: scored_flag, dtype: float64

In [60]:
for col in ['home_club_goals',
            'away_club_goals',
            'goal_diff_abs']:

    print("\n", col)
    print(
        train.groupby(col)['scored_flag']
        .mean()
        .head(15)
    )


 home_club_goals
home_club_goals
0     0.045613
1     0.072981
2     0.099041
3     0.121375
4     0.139977
5     0.155908
6     0.164768
7     0.171897
8     0.203141
9     0.196172
10    0.236246
11    0.189655
12    0.080000
13    0.074074
14    0.166667
Name: scored_flag, dtype: float64

 away_club_goals
away_club_goals
0     0.051496
1     0.081733
2     0.106777
3     0.129668
4     0.144253
5     0.156946
6     0.176661
7     0.197621
8     0.195431
9     0.266355
10    0.234848
11    0.189781
12    0.346939
13    0.226667
15    0.000000
Name: scored_flag, dtype: float64

 goal_diff_abs
goal_diff_abs
0     0.065630
1     0.076071
2     0.092213
3     0.110624
4     0.132916
5     0.152641
6     0.177602
7     0.210891
8     0.250960
9     0.301242
10    0.340659
11    0.448276
12    0.428571
13    0.333333
16    0.000000
Name: scored_flag, dtype: float64


In [61]:
drop_cols = [
    'home_club_goals',
    'away_club_goals',
    'goal_diff_abs'
]

In [62]:
from lightgbm import LGBMClassifier

model2 = LGBMClassifier(
    objective='binary',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model2.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Number of positive: 90143, number of negative: 965707
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.180634 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7743
[LightGBM] [Info] Number of data points in the train set: 1055850, number of used features: 66
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.085375 -> initscore=-2.371463
[LightGBM] [Info] Start training from score -2.371463


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
               n_jobs=-1, num_leaves=63, objective='binary', random_state=42,
               subsample=0.8)

In [63]:
from sklearn.metrics import average_precision_score

preds2 = model2.predict_proba(X_valid)[:,1]

ap2 = average_precision_score(
    y_valid,
    preds2
)

print("Old AP =", 0.3995550743673381)
print("New AP =", ap2)
print("Gain =", ap2 - 0.3995550743673381)

Old AP = 0.3995550743673381
New AP = 0.3318568512232416
Gain = -0.06769822314409651


In [64]:
print(X_train2.columns)

Index(['date', 'season', 'minutes_played', 'yellow_cards', 'red_cards',
       'disciplinary_points', 'home_club_id', 'away_club_id', 'home_club_name',
       'away_club_name', 'home_club_goals', 'away_club_goals', 'home_away',
       'attendance', 'stadium', 'referee', 'name_x', 'country_name',
       'competition_type', 'confederation', 'market_value_before_match', 'age',
       'height_in_cm', 'foot', 'position', 'sub_position',
       'country_of_citizenship', 'international_caps', 'international_goals',
       'highest_market_value_in_eur', 'market_value_ratio', 'avg_xG', 'avg_xA',
       'avg_shots', 'avg_key_passes', 'avg_xGChain', 'avg_xGBuildup',
       'avg_npxG', 'has_understat', 'log_market_value', 'value_pct_of_peak',
       'market_value_tier', 'minutes_ratio', 'full_match_flag', 'starter_flag',
       'substitute_flag', 'card_flag', 'goal_diff_abs', 'age_bucket',
       'prime_age_flag', 'veteran_flag', 'goal_per_cap',
       'has_national_team_experience', 'xG_to_xA_rat

In [65]:
print('player_goal_rate' in X_train.columns)

True


In [66]:
X = train.drop(columns=['scored_flag'])
y = train['scored_flag'].astype(int)

In [67]:
print(X.shape)

(1319813, 65)


In [68]:
print('attacking_index' in train.columns)
print('creative_index' in train.columns)
print('player_goal_rate' in train.columns)

True
True
False


In [69]:
train.drop(
    columns=['attacking_index', 'creative_index'],
    inplace=True
)

print(train.shape)

(1319813, 64)


In [70]:
X = train.drop(columns=['scored_flag']).copy()
y = train['scored_flag'].astype(int)

print(X.shape)

(1319813, 63)


In [71]:
print(train.shape)
print(X.shape)

(1319813, 64)
(1319813, 63)


In [75]:
X['attacking_index'] = (
    X['avg_xG'].fillna(0)
    + X['avg_npxG'].fillna(0)
    + X['avg_shots'].fillna(0)
)

X['creative_index'] = (
    X['avg_xA'].fillna(0)
    + X['avg_key_passes'].fillna(0)
)

In [76]:
print(X[['attacking_index', 'creative_index']].head())

   attacking_index  creative_index
0        25.465710       26.021390
1        18.979778       17.883399
2         0.000000        0.000000
3        20.589999       19.396250
4         0.000000        0.000000


In [78]:
from sklearn.model_selection import GroupKFold

groups = train['name_y'].fillna('Unknown_Player')

gkf = GroupKFold(n_splits=5)

train_idx, valid_idx = next(
    gkf.split(X, y, groups)
)

X_train = X.iloc[train_idx].copy()
X_valid = X.iloc[valid_idx].copy()

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

In [80]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:

    X[col] = X[col].fillna('Missing').astype(str)

    le = LabelEncoder()

    X[col] = le.fit_transform(X[col])

In [81]:
X.dtypes.value_counts()

int64      30
float64    22
bool       13
Name: count, dtype: int64

In [82]:
groups = train['name_y'].fillna('Unknown_Player')

gkf = GroupKFold(n_splits=5)

train_idx, valid_idx = next(
    gkf.split(X, y, groups)
)

X_train = X.iloc[train_idx].copy()
X_valid = X.iloc[valid_idx].copy()

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

In [83]:
model_new.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 90143, number of negative: 965707
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.624153 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7508
[LightGBM] [Info] Number of data points in the train set: 1055850, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.085375 -> initscore=-2.371463
[LightGBM] [Info] Start training from score -2.371463


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
               n_jobs=-1, num_leaves=63, objective='binary', random_state=42,
               subsample=0.8)

In [84]:
from sklearn.metrics import average_precision_score

preds = model_new.predict_proba(X_valid)[:,1]

ap_new = average_precision_score(
    y_valid,
    preds
)

print("Baseline AP =", 0.3995550743673381)
print("New AP =", ap_new)
print("Gain =", ap_new - 0.3995550743673381)

Baseline AP = 0.3995550743673381
New AP = 0.4251877105839332
Gain = 0.02563263621659506


In [85]:
print('attacking_index' in test.columns)
print('creative_index' in test.columns)

True
True


In [86]:
test.dtypes.value_counts()

int64      30
float64    22
bool       13
Name: count, dtype: int64

In [87]:
test_preds = model_new.predict_proba(test)[:, 1]

print(test_preds[:5])

[0.00074996 0.41773172 0.09090226 0.36064706 0.08506516]


In [89]:
import pandas as pd

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/offside-data-thon/sample_submission.csv"
)

sample_submission.head()

,appearance_id,scored_flag
0,3598087_288233,0.5
1,2577964_171424,0.5
2,4676659_855731,0.5
3,3201275_468198,0.5
4,4103610_324804,0.5


In [90]:
submission = sample_submission.copy()

submission['scored_flag'] = test_preds

submission.head()

,appearance_id,scored_flag
0,3598087_288233,0.000750
1,2577964_171424,0.417732
2,4676659_855731,0.090902
3,3201275_468198,0.360647
4,4103610_324804,0.085065


In [91]:
submission.to_csv("submission.csv", index=False)

In [92]:
submission.shape

(565635, 2)

In [93]:
!ls -lh submission.csv

-rw-r--r-- 1 root root 20M Jun  6 08:34 submission.csv


In [95]:
!ls -lh /kaggle/working

total 20M
-rw-r--r-- 1 root root 20M Jun  6 08:34 submission.csv


In [98]:
X['attacking_minus_creative'] = (
    X['attacking_index'] - X['creative_index']
)

test['attacking_minus_creative'] = (
    test['attacking_index'] - test['creative_index']
)

In [102]:
X['attack_creative_ratio'] = (
    X['attacking_index'] /
    (X['creative_index'] + 1)
)

test['attack_creative_ratio'] = (
    test['attacking_index'] /
    (test['creative_index'] + 1)
)

In [104]:
[col for col in X.columns if 'pos' in col.lower()]

['position', 'sub_position']

In [123]:
from catboost import CatBoostClassifier

In [124]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    verbose=0,
    random_seed=42
)

In [125]:
cat_model.fit(X_train, y_train)

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=0)

In [126]:
from sklearn.metrics import average_precision_score

cat_preds = cat_model.predict_proba(X_valid)[:, 1]

cat_ap = average_precision_score(
    y_valid,
    cat_preds
)

print("CatBoost AP =", cat_ap)

CatBoost AP = 0.4274952571707449


In [127]:
lgb_preds = model_new.predict_proba(X_valid)[:, 1]

ensemble_preds = (
    0.5 * lgb_preds +
    0.5 * cat_preds
)

ensemble_ap = average_precision_score(
    y_valid,
    ensemble_preds
)

print("Ensemble AP =", ensemble_ap)

Ensemble AP = 0.42781162839291953


In [128]:
model_new.fit(X, y)

[LightGBM] [Info] Number of positive: 112682, number of negative: 1207131
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.204322 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8292
[LightGBM] [Info] Number of data points in the train set: 1319813, number of used features: 68
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.085377 -> initscore=-2.371432
[LightGBM] [Info] Start training from score -2.371432


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
               n_jobs=-1, num_leaves=63, objective='binary', random_state=42,
               subsample=0.8)

In [129]:
cat_model.fit(X, y)

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=0)

In [130]:
lgb_test_preds = model_new.predict_proba(test)[:, 1]

cat_test_preds = cat_model.predict_proba(test)[:, 1]

In [146]:
final_preds = (
    0.5 * lgb_test_preds +
    0.5 * cat_test_preds
)

In [147]:
print("LightGBM AP :", average_precision_score(y_valid, lgb_preds))
print("CatBoost AP :", average_precision_score(y_valid, cat_preds))
print("Ensemble AP :", average_precision_score(y_valid, ensemble_preds))

LightGBM AP : 0.42519483429154886
CatBoost AP : 0.4274952571707449
Ensemble AP : 0.42781162839291953


In [148]:
for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    preds = w * lgb_preds + (1 - w) * cat_preds
    ap = average_precision_score(y_valid, preds)
    print(f"LGB={w:.1f}, CAT={1-w:.1f} -> AP={ap:.6f}")

LGB=0.0, CAT=1.0 -> AP=0.427495
LGB=0.1, CAT=0.9 -> AP=0.427774
LGB=0.2, CAT=0.8 -> AP=0.427948
LGB=0.3, CAT=0.7 -> AP=0.428015
LGB=0.4, CAT=0.6 -> AP=0.427964
LGB=0.5, CAT=0.5 -> AP=0.427812
LGB=0.6, CAT=0.4 -> AP=0.427529
LGB=0.7, CAT=0.3 -> AP=0.427111
LGB=0.8, CAT=0.2 -> AP=0.426590
LGB=0.9, CAT=0.1 -> AP=0.425964
LGB=1.0, CAT=0.0 -> AP=0.425195


In [149]:
from catboost import CatBoostClassifier

cat_model2 = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.03,
    loss_function='Logloss',
    random_seed=42,
    verbose=0
)

In [150]:
cat_model2.fit(X_train, y_train)

CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=0)

In [151]:
cat2_preds = cat_model2.predict_proba(X_valid)[:,1]

cat2_ap = average_precision_score(
    y_valid,
    cat2_preds
)

print(cat2_ap)

0.42779163396089936


In [152]:
train[['appearance_id','scored_flag']].head()

,appearance_id,scored_flag
0,2335635_20137,False
1,2654328_36235,False
2,4098806_1163778,False
3,2700658_348863,False
4,3016404_240551,True


In [153]:
train['appearance_id'].describe()

count           1319813
unique          1319813
top       2335635_20137
freq                  1
Name: appearance_id, dtype: object

In [154]:
train.groupby('sub_position')['scored_flag'].mean() \
     .sort_values(ascending=False) \
     .head(30)

sub_position
Centre-Forward        0.210135
Second Striker        0.198219
Left Winger           0.131850
Right Winger          0.129926
Attacking Midfield    0.129067
Right Midfield        0.077868
Left Midfield         0.074396
Central Midfield      0.073254
Defensive Midfield    0.044164
Centre-Back           0.040188
Right-Back            0.033788
Left-Back             0.033232
Goalkeeper            0.000162
Name: scored_flag, dtype: float64

In [155]:
print(X['sub_position'].dtype)

int64


In [156]:
train['sub_position'].nunique()

13

In [157]:
player_counts = train['name_y'].value_counts()

print(player_counts.describe())

count    27282.000000
mean        48.376622
std         63.459784
min          1.000000
25%          6.000000
50%         22.000000
75%         66.000000
max        812.000000
Name: count, dtype: float64


In [158]:
train['date'] = pd.to_datetime(train['date'])

train[['name_y', 'date', 'scored_flag']] \
    .sort_values(['name_y', 'date']) \
    .head(20)

,name_y,date,scored_flag
840831,A.J. Soares,2016-07-31,False
1277500,A.J. Soares,2016-08-21,False
64822,AJ Leitch-Smith,2017-09-09,False
763805,AJ Leitch-Smith,2017-09-16,True
1168631,AJ Leitch-Smith,2017-09-30,False
919325,AJ Leitch-Smith,2017-10-14,False
60440,AJ Leitch-Smith,2017-10-21,True
687435,AJ Leitch-Smith,2017-10-25,False
883237,AJ Leitch-Smith,2017-11-04,False
1170965,AJ Leitch-Smith,2017-12-08,False


In [159]:
tmp = train[['name_y', 'date', 'scored_flag']].copy()

tmp = tmp.sort_values(['name_y', 'date'])

tmp['player_matches_before'] = (
    tmp.groupby('name_y')
       .cumcount()
)

tmp[['name_y', 'date', 'player_matches_before']].head(20)

,name_y,date,player_matches_before
840831,A.J. Soares,2016-07-31,0.0
1277500,A.J. Soares,2016-08-21,1.0
64822,AJ Leitch-Smith,2017-09-09,0.0
763805,AJ Leitch-Smith,2017-09-16,1.0
1168631,AJ Leitch-Smith,2017-09-30,2.0
919325,AJ Leitch-Smith,2017-10-14,3.0
60440,AJ Leitch-Smith,2017-10-21,4.0
687435,AJ Leitch-Smith,2017-10-25,5.0
883237,AJ Leitch-Smith,2017-11-04,6.0
1170965,AJ Leitch-Smith,2017-12-08,7.0


In [160]:
tmp = train[['name_y', 'date', 'scored_flag']].copy()

tmp = tmp.sort_values(['name_y', 'date'])

tmp['player_matches_before'] = (
    tmp.groupby('name_y')
       .cumcount()
)

tmp['player_goals_before'] = (
    tmp.groupby('name_y')['scored_flag']
       .cumsum()
       .shift(1)
       .fillna(0)
)

tmp['player_goal_rate_before'] = (
    tmp['player_goals_before'] /
    (tmp['player_matches_before'] + 1)
)

tmp[['name_y',
      'date',
      'scored_flag',
      'player_matches_before',
      'player_goals_before',
      'player_goal_rate_before']].head(20)

,name_y,date,scored_flag,player_matches_before,player_goals_before,player_goal_rate_before
840831,A.J. Soares,2016-07-31,False,0.0,0.0,0.000000
1277500,A.J. Soares,2016-08-21,False,1.0,0.0,0.000000
64822,AJ Leitch-Smith,2017-09-09,False,0.0,0.0,0.000000
763805,AJ Leitch-Smith,2017-09-16,True,1.0,0.0,0.000000
1168631,AJ Leitch-Smith,2017-09-30,False,2.0,1.0,0.333333
919325,AJ Leitch-Smith,2017-10-14,False,3.0,1.0,0.250000
60440,AJ Leitch-Smith,2017-10-21,True,4.0,1.0,0.200000
687435,AJ Leitch-Smith,2017-10-25,False,5.0,2.0,0.333333
883237,AJ Leitch-Smith,2017-11-04,False,6.0,2.0,0.285714
1170965,AJ Leitch-Smith,2017-12-08,False,7.0,2.0,0.250000


In [161]:
tmp['player_goal_rate_before'].describe()

count    1.319811e+06
mean     1.678856e-01
std      1.578991e+00
min      0.000000e+00
25%      0.000000e+00
50%      5.263158e-02
75%      1.307692e-01
max      2.360000e+02
Name: player_goal_rate_before, dtype: float64

In [162]:
tmp.groupby(
    pd.qcut(
        tmp['player_goal_rate_before'],
        q=10,
        duplicates='drop'
    )
)['scored_flag'].mean()

/tmp/ipykernel_58/2387824343.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby(


player_goal_rate_before
(-0.001, 0.01]      0.034007
(0.01, 0.0333]      0.034740
(0.0333, 0.0526]    0.049335
(0.0526, 0.0769]    0.065608
(0.0769, 0.111]     0.089677
(0.111, 0.157]      0.121989
(0.157, 0.231]      0.166302
(0.231, 236.0]      0.228403
Name: scored_flag, dtype: float64

In [163]:
tmp[['scored_flag']].head()
print(tmp['scored_flag'].dtype)

bool


In [164]:
tmp.sort_values(
    'player_goal_rate_before',
    ascending=False
)[[
    'name_y',
    'player_matches_before',
    'player_goals_before',
    'player_goal_rate_before'
]].head(20)

,name_y,player_matches_before,player_goals_before,player_goal_rate_before
468566,Robert Ljubicic,0.0,236.0,236.0
400660,Lionel Mpasi-Nzau,0.0,214.0,214.0
356635,Harry Lewis,0.0,209.0,209.0
1267079,Cristián Borja,0.0,195.0,195.0
1285167,Mohamed Sankoh,0.0,186.0,186.0
759572,Luis Tinoco,0.0,174.0,174.0
61485,Romeo Akachukwu,0.0,166.0,166.0
1040586,Pierre-Emile Højbjerg,0.0,160.0,160.0
1293867,Kyliane Dong,0.0,158.0,158.0
1216440,Antoine Hainaut,0.0,150.0,150.0


In [165]:
tmp = train[['name_y', 'date', 'scored_flag']].copy()

tmp = tmp.sort_values(['name_y', 'date'])

tmp['player_matches_before'] = (
    tmp.groupby('name_y')
       .cumcount()
)

tmp['player_goals_before'] = (
    tmp.groupby('name_y')['scored_flag']
       .transform(lambda x: x.cumsum().shift(1))
       .fillna(0)
)

tmp['player_goal_rate_before'] = (
    tmp['player_goals_before'] /
    (tmp['player_matches_before'] + 1)
)

In [166]:
tmp.sort_values(
    'player_goal_rate_before',
    ascending=False
)[[
    'name_y',
    'player_matches_before',
    'player_goals_before',
    'player_goal_rate_before'
]].head(20)

,name_y,player_matches_before,player_goals_before,player_goal_rate_before
200390,Carlos Bacca,8.0,8.0,0.888889
541251,Carlos Bacca,7.0,7.0,0.875000
72913,David Clarkson,7.0,7.0,0.875000
302660,David Clarkson,6.0,6.0,0.857143
769589,Carlos Bacca,6.0,6.0,0.857143
1192430,Erling Haaland,6.0,6.0,0.857143
1051910,Aurélien Joachim,5.0,5.0,0.833333
171505,Krzysztof Piatek,5.0,5.0,0.833333
476390,David Clarkson,5.0,5.0,0.833333
839955,Erling Haaland,5.0,5.0,0.833333


In [167]:
tmp.groupby(
    pd.qcut(
        tmp['player_goal_rate_before'],
        q=20,
        duplicates='drop'
    )
)['scored_flag'].mean()

/tmp/ipykernel_58/3603125686.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp.groupby(


player_goal_rate_before
(-0.001, 0.0208]    0.034216
(0.0208, 0.0312]    0.038654
(0.0312, 0.0407]    0.042511
(0.0407, 0.0508]    0.052363
(0.0508, 0.0625]    0.059832
(0.0625, 0.0743]    0.067524
(0.0743, 0.0893]    0.078443
(0.0893, 0.106]     0.092475
(0.106, 0.125]      0.107280
(0.125, 0.15]       0.125952
(0.15, 0.181]       0.150034
(0.181, 0.22]       0.170209
(0.22, 0.28]        0.207857
(0.28, 0.889]       0.276901
Name: scored_flag, dtype: float64

In [168]:
train = train.sort_values(['name_y', 'date'])

train['player_matches_before'] = (
    train.groupby('name_y')
         .cumcount()
)

train['player_goals_before'] = (
    train.groupby('name_y')['scored_flag']
         .transform(lambda x: x.cumsum().shift(1))
         .fillna(0)
)

train['player_goal_rate_before'] = (
    train['player_goals_before'] /
    (train['player_matches_before'] + 1)
)

In [169]:
X['player_matches_before'] = train['player_matches_before']
X['player_goal_rate_before'] = train['player_goal_rate_before']

In [170]:
player_stats = train.groupby('name_y').agg({
    'player_matches_before':'max',
    'player_goals_before':'max'
})

player_stats['player_goal_rate_before'] = (
    player_stats['player_goals_before'] /
    (player_stats['player_matches_before'] + 1)
)

In [171]:
test['player_matches_before'] = (
    test['name_y']
    .map(player_stats['player_matches_before'])
    .fillna(0)
)

test['player_goal_rate_before'] = (
    test['name_y']
    .map(player_stats['player_goal_rate_before'])
    .fillna(0)
)

In [172]:
X_test = test.copy()

In [173]:
cat_model.fit(X_train, y_train)

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=0)

In [174]:
from sklearn.metrics import average_precision_score

cat_preds = cat_model.predict_proba(X_valid)[:, 1]

ap = average_precision_score(
    y_valid,
    cat_preds
)

print("AP =", ap)

AP = 0.4274952571707449


In [175]:
print('player_matches_before' in X_train.columns)
print('player_goal_rate_before' in X_train.columns)

False
False


In [176]:
print('player_matches_before' in X.columns)
print('player_goal_rate_before' in X.columns)

True
True


In [177]:
groups = train['name_y'].fillna('Unknown_Player')

gkf = GroupKFold(n_splits=5)

train_idx, valid_idx = next(
    gkf.split(X, y, groups)
)

X_train = X.iloc[train_idx].copy()
X_valid = X.iloc[valid_idx].copy()

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

In [178]:
print('player_matches_before' in X_train.columns)
print('player_goal_rate_before' in X_train.columns)

True
True


In [179]:
cat_model.fit(X_train, y_train)

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=0)

In [180]:
from sklearn.metrics import average_precision_score

cat_preds = cat_model.predict_proba(X_valid)[:, 1]

ap = average_precision_score(
    y_valid,
    cat_preds
)

print("AP =", ap)

AP = 0.4214759779346712


In [181]:
X = X.drop(
    columns=[
        'player_matches_before',
        'player_goal_rate_before'
    ],
    errors='ignore'
)

In [182]:
groups = train['name_y'].fillna('Unknown_Player')

gkf = GroupKFold(n_splits=5)

train_idx, valid_idx = next(
    gkf.split(X, y, groups)
)

X_train = X.iloc[train_idx].copy()
X_valid = X.iloc[valid_idx].copy()

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

In [183]:
cat_model.fit(X_train, y_train)

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=0)

In [184]:
from sklearn.metrics import average_precision_score

cat_preds = cat_model.predict_proba(X_valid)[:, 1]

ap = average_precision_score(
    y_valid,
    cat_preds
)

print("CatBoost AP =", ap)

CatBoost AP = 0.41944516703421697


In [185]:
print(X.shape)
print(X_train.shape)

(1319813, 68)
(1055850, 68)


In [186]:
print('player_matches_before' in X.columns)
print('player_goal_rate_before' in X.columns)

print('player_matches_before' in X_train.columns)
print('player_goal_rate_before' in X_train.columns)

False
False
False
False


In [187]:
X.columns.tolist()

['appearance_id',
 'date',
 'season',
 'minutes_played',
 'yellow_cards',
 'red_cards',
 'disciplinary_points',
 'home_club_id',
 'away_club_id',
 'home_club_name',
 'away_club_name',
 'home_club_goals',
 'away_club_goals',
 'home_away',
 'attendance',
 'stadium',
 'referee',
 'name_x',
 'country_name',
 'competition_type',
 'confederation',
 'market_value_before_match',
 'age',
 'height_in_cm',
 'foot',
 'position',
 'sub_position',
 'country_of_citizenship',
 'international_caps',
 'international_goals',
 'highest_market_value_in_eur',
 'market_value_ratio',
 'avg_xG',
 'avg_xA',
 'avg_shots',
 'avg_key_passes',
 'avg_xGChain',
 'avg_xGBuildup',
 'avg_npxG',
 'has_understat',
 'log_market_value',
 'value_pct_of_peak',
 'market_value_tier',
 'minutes_ratio',
 'full_match_flag',
 'starter_flag',
 'substitute_flag',
 'card_flag',
 'goal_diff_abs',
 'age_bucket',
 'prime_age_flag',
 'veteran_flag',
 'goal_per_cap',
 'has_national_team_experience',
 'xG_to_xA_ratio',
 'finisher_flag',
 'c

In [188]:
print(cat2_ap)

0.42779163396089936


In [189]:
print("cat_model2" in globals())
print("X_test" in globals())

True
True


In [193]:
groups = train['name_y'].fillna('unknown')

# Verify
print("NaN in groups:", groups.isna().sum())

NaN in groups: 0


In [194]:
# Step 1: Diagnose
print("NaN in groups:", groups.isna().sum())
print("NaN in y:", y.isna().sum())
print("\nTop NaN columns in X:")
print(X.isna().sum().sort_values(ascending=False).head(20))

NaN in groups: 0
NaN in y: 0

Top NaN columns in X:
goal_per_cap                   698431
xG_to_xA_ratio                 659447
avg_shots                      631205
avg_key_passes                 631205
avg_npxG                       631205
avg_xGChain                    631205
avg_xA                         631205
avg_xG                         631205
avg_xGBuildup                  631205
attendance                     141222
log_market_value                11702
value_pct_of_peak               11702
market_value_before_match       11702
height_in_cm                    11145
highest_market_value_in_eur      1666
market_value_ratio               1666
age                               675
international_caps                  2
international_goals                 2
appearance_id                       0
dtype: int64


In [195]:
# Step 2: Fix X and X_test
X = X.fillna(0)
X_test = X_test.fillna(0)

# Step 3: Verify
print("NaN in X after fix:", X.isna().sum().sum())
print("NaN in X_test after fix:", X_test.isna().sum().sum())

NaN in X after fix: 0
NaN in X_test after fix: 0


In [197]:
cat2_test_preds = cat_model2.predict_proba(X_test)[:, 1]

In [198]:
submission = pd.DataFrame({
    'appearance_id': test['appearance_id'],
    'scored_flag': cat2_test_preds
})

submission.head()

,appearance_id,scored_flag
0,1215682,0.000379
1,391639,0.316903
2,1860651,0.076514
3,920901,0.224145
4,1503218,0.087313


In [199]:
submission.to_csv(
    "catboost_best_submission.csv",
    index=False
)

In [200]:
!ls -lh catboost_best_submission.csv

-rw-r--r-- 1 root root 16M Jun  6 10:27 catboost_best_submission.csv


In [201]:
cat2_test_preds[:10]

array([0.00037889, 0.31690299, 0.07651435, 0.22414475, 0.08731259,
       0.18445814, 0.00131784, 0.35343261, 0.06915635, 0.00136975])

In [202]:
submission['scored_flag'].describe()

count    565635.000000
mean          0.077791
std           0.118405
min           0.000007
25%           0.002104
50%           0.035393
75%           0.093144
max           0.974555
Name: scored_flag, dtype: float64

In [203]:
print("cat_model2" in globals())
print("X_test" in globals())

True
True


In [204]:
cat_test_preds = cat_model2.predict_proba(X_test)[:, 1]

print(cat_test_preds[:10])

[0.00037889 0.31690299 0.07651435 0.22414475 0.08731259 0.18445814
 0.00131784 0.35343261 0.06915635 0.00136975]


In [205]:
submission = pd.DataFrame({
    "appearance_id": test["appearance_id"],
    "scored_flag": cat_test_preds
})

submission.head()

,appearance_id,scored_flag
0,1215682,0.000379
1,391639,0.316903
2,1860651,0.076514
3,920901,0.224145
4,1503218,0.087313


In [206]:
submission.to_csv(
    "final_submission.csv",
    index=False
)

In [207]:
!ls -lh final_submission.csv

-rw-r--r-- 1 root root 16M Jun  6 10:34 final_submission.csv


In [208]:
train['has_understat'].value_counts(normalize=True)

has_understat
True     0.521747
False    0.478253
Name: proportion, dtype: float64

In [209]:
train.groupby('has_understat')['scored_flag'].mean()

has_understat
False    0.076982
True     0.093073
Name: scored_flag, dtype: float64

In [210]:
for col in [
    'avg_xG',
    'avg_xA',
    'avg_shots',
    'avg_key_passes',
    'avg_npxG'
]:
    print(col)
    print(train.groupby(train[col].isna())['scored_flag'].mean())
    print()

avg_xG
avg_xG
False    0.093073
True     0.076981
Name: scored_flag, dtype: float64

avg_xA
avg_xA
False    0.093073
True     0.076981
Name: scored_flag, dtype: float64

avg_shots
avg_shots
False    0.093073
True     0.076981
Name: scored_flag, dtype: float64

avg_key_passes
avg_key_passes
False    0.093073
True     0.076981
Name: scored_flag, dtype: float64

avg_npxG
avg_npxG
False    0.093073
True     0.076981
Name: scored_flag, dtype: float64



In [211]:
X.dtypes.value_counts()

int64      30
float64    25
bool       13
Name: count, dtype: int64

In [212]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(len(cat_cols))
print(cat_cols)

0
[]


In [214]:
print("Features in current X:", len(X.columns))

print(
    "Features in model:",
    len(cat_model2.feature_names_)
)

Features in current X: 68
Features in model: 65


In [215]:
cat_model2.feature_names_[:20]

['appearance_id',
 'date',
 'season',
 'minutes_played',
 'yellow_cards',
 'red_cards',
 'disciplinary_points',
 'home_club_id',
 'away_club_id',
 'home_club_name',
 'away_club_name',
 'home_club_goals',
 'away_club_goals',
 'home_away',
 'attendance',
 'stadium',
 'referee',
 'name_x',
 'country_name',
 'competition_type']

In [216]:
X.columns[:20].tolist()

['appearance_id',
 'date',
 'season',
 'minutes_played',
 'yellow_cards',
 'red_cards',
 'disciplinary_points',
 'home_club_id',
 'away_club_id',
 'home_club_name',
 'away_club_name',
 'home_club_goals',
 'away_club_goals',
 'home_away',
 'attendance',
 'stadium',
 'referee',
 'name_x',
 'country_name',
 'competition_type']

In [217]:
current_cols = set(X.columns)
model_cols = set(cat_model2.feature_names_)

print("Extra columns in current X:")
print(current_cols - model_cols)

print("\nMissing columns:")
print(model_cols - current_cols)

Extra columns in current X:
{'attack_creative_ratio', 'attacking_plus_creative', 'attacking_minus_creative'}

Missing columns:
set()


In [218]:
X_65 = X.drop(
    columns=[
        'attack_creative_ratio',
        'attacking_plus_creative',
        'attacking_minus_creative'
    ]
)

print(X_65.shape)

(1319813, 65)


In [219]:
X_exp = X_65.copy()

X_exp['attacking_minus_creative'] = (
    X['attacking_minus_creative']
)

print(X_exp.shape)

(1319813, 66)


In [220]:
groups = train['name_y'].fillna('Unknown_Player')

gkf = GroupKFold(n_splits=5)

train_idx, valid_idx = next(
    gkf.split(X_exp, y, groups)
)

X_train_exp = X_exp.iloc[train_idx].copy()
X_valid_exp = X_exp.iloc[valid_idx].copy()

y_train_exp = y.iloc[train_idx]
y_valid_exp = y.iloc[valid_idx]

print(X_train_exp.shape)
print(X_valid_exp.shape)

(1055850, 66)
(263963, 66)


In [221]:
from catboost import CatBoostClassifier

cat_exp = CatBoostClassifier(
    depth=8,
    iterations=1000,
    learning_rate=0.03,
    loss_function='Logloss',
    random_seed=42,
    verbose=0
)

cat_exp.fit(
    X_train_exp,
    y_train_exp
)

CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=0)

In [222]:
from sklearn.metrics import average_precision_score

exp_preds = cat_exp.predict_proba(X_valid_exp)[:, 1]

exp_ap = average_precision_score(
    y_valid_exp,
    exp_preds
)

print("Experiment AP =", exp_ap)

Experiment AP = 0.4208567907161281


In [223]:
print(cat2_ap)

0.42779163396089936


In [224]:
sub_strength = train.groupby(
    'sub_position'
)['scored_flag'].mean()

print(sub_strength.sort_values(ascending=False))

sub_position
Centre-Forward        0.210135
Second Striker        0.198219
Left Winger           0.131850
Right Winger          0.129926
Attacking Midfield    0.129067
Right Midfield        0.077868
Left Midfield         0.074396
Central Midfield      0.073254
Defensive Midfield    0.044164
Centre-Back           0.040188
Right-Back            0.033788
Left-Back             0.033232
Goalkeeper            0.000162
Name: scored_flag, dtype: float64


In [225]:
print(X['sub_position'].nunique())

14


In [226]:
train['sub_position'].value_counts()

sub_position
Centre-Back           243757
Centre-Forward        186237
Central Midfield      163992
Defensive Midfield    126619
Right-Back            108263
Attacking Midfield     95609
Left-Back              92651
Goalkeeper             92638
Left Winger            88699
Right Winger           85333
Right Midfield         12457
Second Striker         11906
Left Midfield          10592
Name: count, dtype: int64

In [227]:
print("model_new" in globals())

True


In [228]:
print("cat_model2" in globals())

True
